# 04.4 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/04-encoding-categories/exercises.ipynb)

Run the setup cell first.

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail()                                   # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

def customer_table(cutoff, months=3):
    """One row per customer: features from before `cutoff`, churned = no order in the next `months`."""
    cutoff = pd.Timestamp(cutoff)
    past = sales[sales["InvoiceDate"] < cutoff]
    future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < cutoff + pd.DateOffset(months=months))]
    c = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                        products=("StockCode", "nunique"), country=("Country", "first"),
                                        first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
    c["tenure_days"] = (cutoff - c["first_order"]).dt.days
    c["days_since_last"] = (cutoff - c["last_order"]).dt.days
    came_back = future.groupby("Customer ID")["Revenue"].sum().reindex(c.index).fillna(0) > 0
    c["churned"] = (~came_back).astype(int)
    return c.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
cust = customer_table("2011-09-01")
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

six = pd.DataFrame({"customer": ["A", "B", "C", "D", "E", "F"],
                    "country":  ["UK", "DE", "FR", "UK", "FR", "DE"],
                    "plan":     ["S", "L", "M", "M", "S", "L"]})
six

ten = pd.DataFrame({"country": ["UK", "UK", "UK", "UK", "DE", "DE", "FR", "FR", "FR", "NL"],
                    "churned": [1, 0, 0, 1, 0, 0, 1, 0, 1, 1]})

past = sales[sales["InvoiceDate"] < "2011-09-01"]
by_item = past.groupby(["Customer ID", "StockCode"])["Quantity"].sum().reset_index()
fav = by_item.sort_values("Quantity").groupby("Customer ID").tail(1).set_index("Customer ID")["StockCode"]
cust["fav_product"] = fav.reindex(cust.index)       # the product each customer bought most units of

from sklearn.model_selection import train_test_split
tr, te = train_test_split(cust, test_size=0.25, random_state=42, stratify=cust["churned"])
y_tr, y_te = tr["churned"], te["churned"]
base = ["days_since_last", "orders"]
print(f"{cust['country'].nunique()} countries, {cust['fav_product'].nunique()} different favourite products")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def fit_score(A, B):
    """Scale, fit a logistic regression on A (train), return (train acc, test acc)."""
    sc = StandardScaler().fit(A)
    m = LogisticRegression(max_iter=2000).fit(sc.transform(A), y_tr)
    return m.score(sc.transform(A), y_tr), m.score(sc.transform(B), y_te)

3,317 customers, churn rate 43.6%


36 countries, 951 different favourite products


## Exercise 1: by hand

Add a seventh customer G (country "ES", plan "XL") to `six`. What does each encoder (alphabetical label, ordinal with S/M/L, one-hot) do with G if it was fitted on the first six rows? Then try it in code with `handle_unknown` settings that do not crash.

In [2]:
# your code here

## Exercise 2: smoothing strength

In the ten-row table, compute the smoothed target encoding for every country with $m = 0, 2, 10$. What happens to NL as $m$ grows, and what does $m = 0$ correspond to?

In [3]:
# your code here

## Exercise 3: the TargetEncoder trap

Encode `fav_product` with `TargetEncoder` but call `.fit(tr[["fav_product"]], y_tr)` and then `.transform(tr[["fav_product"]])` for the training rows. Compare train and test accuracy with the correct `fit_transform` version.

In [4]:
# your code here

## Exercise 4: trees and label encoding

Train a `DecisionTreeClassifier(max_depth=4)` on `base` + alphabetically label-encoded country, and on `base` + one-hot country. Is label encoding as harmful for a tree as for a linear model? Why?

In [5]:
# your code here